# Modeling

It is the main nootebook related to building prediction models for https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv dataset.

Previous notbook related to data preprocessing and feature engineering - https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb


## Notebook initialization

In [1]:
# Notebook initialization
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# print("Project root:", ROOT)

## Imports & Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


from src.data import load_raw_data

raw_df = load_raw_data()
raw_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
target_col = "y"

## Split Data Train & Validation

In [4]:
from src.data import split_train_val, split_X_y

train_df, val_df = split_train_val(raw_df, stratify_col=target_col)

X_train, X_val, y_train, y_val = split_X_y(train_df, val_df, target_col)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((32950, 20), (8238, 20), (32950,), (8238,))

## Experiments Preparation & Results Tracking

In this section we set up a simple framework for running and tracking modeling experiments.

We reuse the `build_pipeline` function implemented in the [previous notebook](https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb) to construct configurable preprocessing and modeling pipelines.

An empty `results_df` DataFrame is created to store the results of all experiments performed in this notebook.

To automate experiment tracking, we implement an `experiment_logger` decorator that logs model parameters, pipeline configuration, and evaluation metrics (F1 and AUROC for train/test).

The `train_pipeline` function builds and fits the pipeline. Since it is decorated with `experiment_logger`, each call automatically records the experiment results in `results_df`.

In [5]:
from src.pipelining import build_pipeline
# Global experiment table
results_df = pd.DataFrame()
experiment_counter = 0

In [6]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def experiment_logger(func):

    def wrapper(*args, **kwargs):
        global results_df
        global experiment_counter

        start = time.time()

        # Run training function
        pipe, X_train, X_val, y_train, y_val, pipeline_params = func(*args, **kwargs)

        fit_time = time.time() - start

        # Predictions
        y_train_pred = pipe.predict(X_train)
        y_val_pred = pipe.predict(X_val)

        y_train_prob = pipe.predict_proba(X_train)[:, 1]
        y_val_prob = pipe.predict_proba(X_val)[:, 1]

        train_f1 = f1_score(y_train, y_train_pred, pos_label="yes")
        val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")

        train_auc = roc_auc_score(y_train, y_train_prob)
        val_auc = roc_auc_score(y_val, y_val_prob)

        model = pipe.named_steps["classifier"]

        # ------------------------
        # model hyperparameters
        # ------------------------
        default_model = model.__class__()
        params = model.get_params()
        default_params = default_model.get_params()

        model_params = {
            f"model__{k}": v
            for k, v in params.items()
            if k in default_params and v != default_params[k]
        }

        # ------------------------
        # pipeline params
        # ------------------------
        pipe_params = {
            f"pipe__{k}": v
            for k, v in pipeline_params.items()
            if k != "model"
        }

        # keep features count
        n_features =len(pipe.named_steps["preprocessing"].get_feature_names_out())

        row = {
            "experiment_id": experiment_counter,
            "model_name": model.__class__.__name__,
            "fit_time": fit_time,
            "train_f1": train_f1,
            "val_f1": val_f1,
            "train_auroc": train_auc,
            "val_auroc": val_auc,
            "n_features": n_features,
        }

        row.update(pipe_params)
        row.update(model_params)

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

        experiment_counter += 1

        return pipe

    return wrapper

In [7]:
@experiment_logger
def train_pipeline(X_train, X_val, y_train, y_val, pipeline_params):

    pipe = build_pipeline(**pipeline_params)

    pipe.fit(X_train, y_train)

    return pipe, X_train, X_val, y_train, y_val, pipeline_params

## Base Models

Use LogisticRegression, KNeighborsClassifier and DecisionTreeClassifier as base models experiments

### Baseline models
Train few baseline models with default model and pipeline params

In [8]:
base_models = [
    LogisticRegression(solver="liblinear", random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(max_leaf_nodes=100, random_state=42), # Limit max_leaf_nodes=100 to avoid overfitting
]

for base_model in base_models:
    train_pipeline(X_train, X_val, y_train, y_val,
    {
        "model": base_model
    }
)

In [9]:
results_df.sort_values("val_f1", ascending=False).head()

,experiment_id,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,n_features,model__random_state,model__solver,model__max_leaf_nodes
2,2,DecisionTreeClassifier,0.172427,0.452679,0.407924,0.795160,0.790662,59,42.0,NaN,100.0
1,1,KNeighborsClassifier,0.090533,0.488967,0.383562,0.926449,0.746241,59,NaN,NaN,NaN
0,0,LogisticRegression,0.177044,0.339418,0.337408,0.792315,0.800625,59,42.0,liblinear,NaN


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Conclusions from baseline models:**

- The baseline **DecisionTreeClassifier** achieves the best **validation F1 score**, indicating thebest performance on the classification objective among baseline models.
- The baseline **LogisticRegression** achieves the best **validation AUROC**, which is expected since logistic regression produces well-calibrated probability estimates.
- The **KNeighborsClassifier** shows the highest scores on the **training set** but noticeably worse performance on the **validation set**, suggesting a tendency to **overfit** the training data.

</span>

### Logistic Regression

#### Feature Statistical Significance

In this section, we explore the **statistical significance of features** for the baseline Logistic Regression model using `statsmodels.Logit`.

This allows us to:

- Estimate **coefficient significance** (p-values) for each feature  
- Identify which features **increase or decrease** the probability of the target outcome  
- Gain insights into the **most influential predictors** in our model  
- Verify whether the assumptions made in the **EDA step** were correct

Features with **p-values below 0.05** are considered statistically significant.

In [10]:
import statsmodels.api as sm

# STEP 1 - train pipeline avoiding any multicollinearity or other stuff that might spoil statistics
pipe_lr = train_pipeline(X_train, X_val, y_train, y_val,
    {
        "poly_degree": 1,                   # no poly features
        "drop_cols": None,                  # include all cols here (except of "duration")
        "pdays_transform_mode": "binary",   # "binary" to avoid multicollinearity and all p-values 1
        "drop_cats": {                      # drop at least one value per each category to avoid multicollinearity and all p-values 1
            "job": ["unknown"],
            "marital": ["unknown"],
            "education": ["unknown", "illiterate"],
            "default": ["no"],
            "housing": ["yes"],
            "loan": ["yes"],
            "contact": ["telephone"],
            "month": ["dec"],
            "day_of_week": ["fri"],
            "poutcome": ["nonexistent"],
        },
        "model": base_models[0],            # LogisticRegression(solver="liblinear", random_state=42)
    }
)

# STEP 2 - get transformet input data and feature names
X_train_transformed = pipe_lr[:-1].transform(X_train)
feature_names = pipe_lr.named_steps["preprocessing"].get_feature_names_out()
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

# STEP 3 - fit sm.Logit for calculating statistics
X_train_df_const = sm.add_constant(X_train_df)
logit_model = sm.Logit(y_train.map({"no":0, "yes":1}), X_train_df_const)
stats_result_lr = logit_model.fit()

# STEP 4 - print all statistics
print(stats_result_lr.summary())

Optimization terminated successfully.
         Current function value: 0.277029
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                32950
Model:                          Logit   Df Residuals:                    32899
Method:                           MLE   Df Model:                           50
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                  0.2131
Time:                        10:49:14   Log-Likelihood:                -9128.1
converged:                       True   LL-Null:                       -11599.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                 -1.6897      0.484    

In [11]:
# STEP 5 - print features sorted by significance (p-value) - round to 3 decimal and sort by raw p-value
significance_df = pd.DataFrame({
    "feature": stats_result_lr.params.index,
    "coef": stats_result_lr.params.values.round(3),
    "p_value": stats_result_lr.pvalues.values.round(3),
    "p_value_raw": stats_result_lr.pvalues.values,
})

significance_df = significance_df.sort_values("p_value_raw")

significance_df[["feature", "coef", "p_value"]]

,feature,coef,p_value
46,numeric__emp.var.rate,-2.391,0.000
26,cat__contact_cellular,0.751,0.000
47,numeric__cons.price.idx,1.285,0.000
30,cat__month_jun,-1.264,0.000
32,cat__month_may,-0.958,0.000
33,cat__month_nov,-0.956,0.000
31,cat__month_mar,0.974,0.000
48,numeric__cons.conf.idx,0.149,0.000
40,cat__poutcome_failure,-0.392,0.000
21,cat__default_risk,-0.238,0.000


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Statistical Significance – Conclusions**

- Most **socio-economic context features** are statistically significant, except for `euribor3m`. This aligns with our EDA assumption that `euribor3m` is highly correlated with other socio-economic features and does not add new information to the model.  
- The features `previous` and `pdays` do not show statistical significance in the Logistic Regression model. This may be due to strong **class imbalance** (e.g., only ~4% of rows have `pdays != 999`). They might still be useful for **tree-based models**.  
- Numeric `age` does not appear significant as-is; using **binned age groups** may improve linear model performance, while tree-based models can handle raw numeric values.  
- Most **month** and **day_of_week** values are significant, suggesting **time-related effects** influence the target.  
- `housing` and `loan` remain insignificant, consistent with the EDA assumptions.  
- No statistically significant effects are observed for categorical variables `education`, `marital`, or `job` in the Logistic Regression model. This may be due to **high cardinality**.  
- Features such as `poutcome_failure`, `default_risk`, `contact_cellular`, and `campaign` are significant predictors of the target.

Overall, these results largely confirm the patterns observed in EDA and provide guidance for **feature selection and transformations** for linear models.
</span>

#### Drop Columns

Experiment with dropping additional columns for the Logistic Regression model based on the statistical significance analysis above.

In [12]:
# based on stats significans experiment above
drop_cols_options = [
    ["loan", "housing"],
    ["loan", "housing", "euribor3m"],
    ["loan", "housing", "euribor3m", "job"],
    ["loan", "housing", "euribor3m", "pdays"],
    ["loan", "housing", "euribor3m", "age"],
    ["loan", "housing", "euribor3m", "previous"],
    ["loan", "housing", "euribor3m", "job", "marital"],
    ["loan", "housing", "euribor3m", "job", "education", "marital"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "age"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays"],
    ["loan", "housing", "euribor3m", "pdays", "age"],
    ["loan", "housing", "euribor3m", "pdays", "age", "previous"],
    ["loan", "housing", "euribor3m", "job", "pdays", "age", "previous"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous", "age"],
]

for drop_cols in drop_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "drop_cols": drop_cols,
            "model": base_models[0],
        }
    )

In [13]:
def show_results_df(
    model_name="LogisticRegression",
    sort_by="val_f1",
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10,
    max_experiment_id=np.inf,   # to reproduce output in case of re-run and do not show stored further experiment
):
    # Show full column content without truncation
    with pd.option_context("display.max_colwidth", None):
        display(
            results_df[
                (results_df["model_name"] == model_name) &
                (results_df["experiment_id"] <= max_experiment_id)
            ].sort_values(sort_by, ascending=False)[show_cols].head(show_count)
        )

show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__drop_cols", "n_features"],
    max_experiment_id=18
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__drop_cols,n_features
17,LogisticRegression,0.087784,0.509921,0.542418,0.933289,0.942179,"[loan, housing, euribor3m, job, education, marital, pdays, previous]",28
13,LogisticRegression,0.090003,0.509921,0.542418,0.933292,0.942181,"[loan, housing, euribor3m, job, education, marital, pdays]",29
11,LogisticRegression,0.118263,0.510652,0.542305,0.933514,0.942194,"[loan, housing, euribor3m, job, education, marital]",33
9,LogisticRegression,0.145252,0.515789,0.541333,0.934378,0.942431,"[loan, housing, euribor3m, previous]",52
12,LogisticRegression,0.105266,0.511313,0.540721,0.933504,0.942200,"[loan, housing, euribor3m, job, education, marital, age]",32
8,LogisticRegression,0.150651,0.515077,0.540613,0.934420,0.942380,"[loan, housing, euribor3m, age]",52
18,LogisticRegression,0.085293,0.509434,0.539491,0.933285,0.942190,"[loan, housing, euribor3m, job, education, marital, pdays, previous, age]",27
5,LogisticRegression,0.155848,0.514747,0.539281,0.934417,0.942376,"[loan, housing, euribor3m]",53
16,LogisticRegression,0.109076,0.511074,0.539130,0.933648,0.942146,"[loan, housing, euribor3m, job, pdays, age, previous]",36
10,LogisticRegression,0.127905,0.514806,0.538770,0.933857,0.942134,"[loan, housing, euribor3m, job, marital]",39


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Selection Impact – Logistic Regression**

- Excluding the features that were not statistically significant leads to a **noticeable improvement in validation metrics** for the Logistic Regression model.  
- Interestingly, including `age` still provides a **slight improvement** in F1 and AUROC on the validation set, suggesting they may carry some predictive signal even if not statistically significant in the linear model.

</span>

#### Polynomial Degree

Create polynomial degree features for the Logistic Regression experiment.

In [14]:
poly_degree_options = [1,2,3,4]

for poly_degree in poly_degree_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "poly_degree": poly_degree,
            "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
            "model": base_models[0],
        }
    )

/Users/maksymstefanko/ML/ML-love/ml-bank-additional-project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [15]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__poly_degree", "n_features"],
    show_count=5,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__poly_degree,n_features
21,LogisticRegression,7.676267,0.567655,0.587292,0.942538,0.948188,3.0,140
22,LogisticRegression,45.677779,0.587458,0.579963,0.946096,0.948698,4.0,350
20,LogisticRegression,0.391889,0.543705,0.566061,0.937656,0.945405,2.0,56
19,LogisticRegression,0.092313,0.509921,0.542418,0.933289,0.942179,1.0,28
17,LogisticRegression,0.087784,0.509921,0.542418,0.933289,0.942179,NaN,28


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Polynomial Feature Expansion – Conclusions**

- The **4th-degree polynomial** does not provide any improvement in validation metrics compared to the **3rd-degree polynomial**.
- The **3rd-degree polynomial** achieves the best **validation F1 and AUROC scores**.
- The **2nd-degree polynomial** offers the best **trade-off between performance (F1/AUROC) and training time**.

</span>

#### Age Binning

Experiment with grouping the numeric `age` feature into categorical groups.

In [16]:
poly_degree_options = [1,2,3]
age_bin_mode_options = [None, "group", "range"]

for age_bin_mode in age_bin_mode_options:
    for poly_degree in poly_degree_options:
        train_pipeline(X_train, X_val, y_train, y_val,
                {
                    "poly_degree": poly_degree,
                    "age_bin_mode": age_bin_mode,
                    "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
                    "model": base_models[0],
                }
            )

In [17]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__poly_degree", "pipe__age_bin_mode", "n_features"],
    show_count=10,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__poly_degree,pipe__age_bin_mode,n_features
28,LogisticRegression,3.462423,0.568751,0.588600,0.941825,0.948018,3.0,group,107
31,LogisticRegression,3.070229,0.566907,0.588089,0.941927,0.948146,3.0,range,110
25,LogisticRegression,7.263534,0.567655,0.587292,0.942538,0.948188,3.0,None,140
21,LogisticRegression,7.676267,0.567655,0.587292,0.942538,0.948188,3.0,NaN,140
22,LogisticRegression,45.677779,0.587458,0.579963,0.946096,0.948698,4.0,NaN,350
30,LogisticRegression,0.273124,0.541351,0.571970,0.937640,0.945530,2.0,range,54
27,LogisticRegression,0.251855,0.540942,0.569804,0.937477,0.945338,2.0,group,51
24,LogisticRegression,0.391648,0.543705,0.566061,0.937656,0.945405,2.0,None,56
20,LogisticRegression,0.391889,0.543705,0.566061,0.937656,0.945405,2.0,NaN,56
19,LogisticRegression,0.092313,0.509921,0.542418,0.933289,0.942179,1.0,NaN,28


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Age Binning – Conclusions**

- Grouping the numeric `age` feature into **three categories (young / middle / old)** improves the model's performance.  
- Creating **more detailed age ranges** also improves performance, but the **three-category grouping appears to provide the best balance**.

</span>

#### Cyclical Encoding for Calendar Columns

Experiment with different encoding strategies for the `month` and `day_of_week` features.

In [18]:
calendar_cols_mode_options = ["onehot", "num", "cyclical"]

for calendar_cols_mode in calendar_cols_mode_options:
    for poly_degree in poly_degree_options:
        train_pipeline(X_train, X_val, y_train, y_val,
                {
                    "calendar_cols_mode": calendar_cols_mode,
                    "poly_degree": poly_degree,
                    "age_bin_mode": "group",
                    "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
                    "model": base_models[0],
                }
            )

In [29]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__calendar_cols_mode", "pipe__poly_degree", "n_features"],
    show_count=10,
    max_experiment_id=40
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__calendar_cols_mode,pipe__poly_degree,n_features
40,LogisticRegression,20.363211,0.583499,0.590265,0.944606,0.949474,cyclical,3.0,294
34,LogisticRegression,3.516036,0.568751,0.588600,0.941825,0.948018,onehot,3.0,107
28,LogisticRegression,3.462423,0.568751,0.588600,0.941825,0.948018,NaN,3.0,107
31,LogisticRegression,3.070229,0.566907,0.588089,0.941927,0.948146,NaN,3.0,110
25,LogisticRegression,7.263534,0.567655,0.587292,0.942538,0.948188,NaN,3.0,140
21,LogisticRegression,7.676267,0.567655,0.587292,0.942538,0.948188,NaN,3.0,140
22,LogisticRegression,45.677779,0.587458,0.579963,0.946096,0.948698,NaN,4.0,350
37,LogisticRegression,10.686873,0.576245,0.579370,0.942820,0.948660,num,3.0,173
30,LogisticRegression,0.273124,0.541351,0.571970,0.937640,0.945530,NaN,2.0,54
36,LogisticRegression,0.443412,0.543046,0.571967,0.938030,0.945744,num,2.0,53


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Cyclical Encoding – Conclusions**

- Cyclical encoding of the `month` and `day_of_week` features improves the model's performance.
- However, when using polynomial features (degree > 1), cyclical encoding increases the total number of generated features.

</span>

#### Social and Economic Context Attributes

Experiment with binning socio-economic context features into categorical groups.

In [20]:
soceco_bin_cols_options = [
    ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "nr.employed"],
    ["emp.var.rate", "cons.price.idx", "cons.conf.idx"],
    ["emp.var.rate", "cons.price.idx", "nr.employed"],
    ["emp.var.rate", "cons.conf.idx", "nr.employed"],
    ["cons.price.idx", "cons.conf.idx", "nr.employed"],
    ["emp.var.rate", "nr.employed"],
    ["cons.price.idx", "cons.conf.idx"],
]

for soceco_bin_cols in soceco_bin_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
            {
                "soceco_bin_cols": soceco_bin_cols,
                "calendar_cols_mode": "cyclical",
                "poly_degree": 3,
                "age_bin_mode": "group",
                "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
                "model": base_models[0],
            }
        )

In [21]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__soceco_bin_cols", "n_features"],
    show_count=10,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__soceco_bin_cols,n_features
46,LogisticRegression,10.690498,0.576769,0.596035,0.943802,0.950296,"[emp.var.rate, nr.employed]",178
47,LogisticRegression,11.076711,0.575313,0.595194,0.943430,0.949228,"[cons.price.idx, cons.conf.idx]",180
40,LogisticRegression,20.363211,0.583499,0.590265,0.944606,0.949474,NaN,294
34,LogisticRegression,3.516036,0.568751,0.588600,0.941825,0.948018,NaN,107
28,LogisticRegression,3.462423,0.568751,0.588600,0.941825,0.948018,NaN,107
31,LogisticRegression,3.070229,0.566907,0.588089,0.941927,0.948146,NaN,110
42,LogisticRegression,4.547441,0.569108,0.587660,0.942448,0.948354,"[emp.var.rate, cons.price.idx, cons.conf.idx]",137
21,LogisticRegression,7.676267,0.567655,0.587292,0.942538,0.948188,NaN,140
25,LogisticRegression,7.263534,0.567655,0.587292,0.942538,0.948188,NaN,140
45,LogisticRegression,5.873084,0.566026,0.586782,0.942947,0.949457,"[cons.price.idx, cons.conf.idx, nr.employed]",138


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Socio-Economic Context Features – Conclusions**

- Binning some (but not all) socio-economic context features into categorical groups improves model performance. Binning `[emp.var.rate, nr.employed]` or `[cons.price.idx, cons.conf.idx]` appears to be optimal. When more than two features are binned, performance tends to drop.  
- Binning socio-economic context features also reduces the total number of generated features when using polynomial features (degree > 1), helping control model complexity.

</span>

#### Hyperparameter Tuning with GridSearchCV

In [38]:
from sklearn.model_selection import GridSearchCV

# create a pipeline based on previous experiments
pipe_lr_gs = build_pipeline(
    soceco_bin_cols=["emp.var.rate", "nr.employed"],
    calendar_cols_mode="cyclical",
    poly_degree=3,
    age_bin_mode="group",
    drop_cols=["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
    model=LogisticRegression(solver="liblinear", random_state=42)
)

# grid params to test
param_grid = {
    "classifier__C": [0.1, 0.5, 1, 2, 10],
    "classifier__l1_ratio": [0, 1],
}

# prepare grid_search
grid_search = GridSearchCV(
    estimator=pipe_lr_gs,
    param_grid=param_grid,
    scoring="f1",  # or "roc_auc"
    n_jobs=-1,     # use all CPUs
    cv=4,          # 4-fold cross-validation
    verbose=0
)

# map target to 1/0 for proper scoring="f1"
y_train_bin = y_train.map({"no": 0, "yes": 1})
y_val_bin = y_val.map({"no": 0, "yes": 1})

In [39]:
%%time
grid_search.fit(X_train, y_train_bin)

/Users/maksymstefanko/ML/ML-love/ml-bank-additional-project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/maksymstefanko/ML/ML-love/ml-bank-additional-project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/maksymstefanko/ML/ML-love/ml-bank-additional-project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


CPU times: user 4min 36s, sys: 1.87 s, total: 4min 38s
Wall time: 14min 1s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 0.5, ...], 'classifier__l1_ratio': [0, 1]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",4
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displaye

In [ ]:
# best found model evaluation
best_pipe = grid_search.best_estimator_
y_val_pred = best_pipe.predict(X_val)
y_val_prob = best_pipe.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val_bin, y_val_pred)
val_auroc = roc_auc_score(y_val_bin, y_val_prob)

val_f1, val_auroc

(0.5939243645381277, 0.9502447786452191)

In [44]:
# run train_pipeline with best found hyperparams
train_pipeline(X_train, X_val, y_train, y_val,
    {
        "soceco_bin_cols": soceco_bin_cols,
        "calendar_cols_mode": "cyclical",
        "poly_degree": 3,
        "age_bin_mode": "group",
        "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
        "model": LogisticRegression(solver="liblinear", l1_ratio=1, C=2, random_state=42)
    }
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineering', ...), ('cyclical_encoding', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols_initial', ...), ('campaign_prev', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['loan', 'housing', ...]"
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators 

In [47]:
show_results_df(
    show_cols=["model_name", "model__l1_ratio", "model__C", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10,
)

,model_name,model__l1_ratio,model__C,fit_time,train_f1,val_f1,train_auroc,val_auroc
46,LogisticRegression,NaN,NaN,10.690498,0.576769,0.596035,0.943802,0.950296
47,LogisticRegression,NaN,NaN,11.076711,0.575313,0.595194,0.943430,0.949228
48,LogisticRegression,1.0,2.0,136.325032,0.573568,0.591497,0.943432,0.949245
40,LogisticRegression,NaN,NaN,20.363211,0.583499,0.590265,0.944606,0.949474
34,LogisticRegression,NaN,NaN,3.516036,0.568751,0.588600,0.941825,0.948018
28,LogisticRegression,NaN,NaN,3.462423,0.568751,0.588600,0.941825,0.948018
31,LogisticRegression,NaN,NaN,3.070229,0.566907,0.588089,0.941927,0.948146
42,LogisticRegression,NaN,NaN,4.547441,0.569108,0.587660,0.942448,0.948354
21,LogisticRegression,NaN,NaN,7.676267,0.567655,0.587292,0.942538,0.948188
25,LogisticRegression,NaN,NaN,7.263534,0.567655,0.587292,0.942538,0.948188


<span style="background-color: #4FC3F7; display:block; padding:10px">

**GridSearchCV – Conclusions**

- The best pipeline found with GridSearchCV ranks only **3rd overall** among all experiments.  
- Due to **smaller regularization** and a **3rd-degree polynomial**, it has a **significantly higher training time**.  
- Therefore, it is reasonable to continue using the **best Logistic Regression pipeline identified in previous experiments**, which provides a better trade-off between performance and efficiency.

</span>

#### Save Logistic Regression Pipelin

In [ ]:
# pipeline previously created for GridSearchCV but not fitted yet
pipe_lr_gs.fit(X_train, y_train)
pipe_lr_gs

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineering', ...), ('cyclical_encoding', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols_initial', ...), ('campaign_prev', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['loan', 'housing', ...]"
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators 

In [ ]:
# save to "models" folder
import joblib
from src.config import MODELS_DIR

joblib.dump(
    pipe_lr_gs,
    MODELS_DIR / "log_reg_model_pipeline.joblib"
);

In [ ]:
# smoke test if all had been saved properly and ca nbe used in future
pipe_lr_smoke = joblib.load(MODELS_DIR / "log_reg_model_pipeline.joblib")

y_val_pred = pipe_lr_smoke.predict(X_val)
y_val_prob = pipe_lr_smoke.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")
val_auroc = roc_auc_score(y_val, y_val_prob)

val_f1, val_auroc

(0.5960346964064436, 0.9502957097032878)